<a href="https://colab.research.google.com/github/edusgr/EDP-II/blob/main/UBER_REGRESION_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto final de Econometría

## Modelo de regresión lineal múltiple aplicado a tarifas de UberX

El objetivo de este proyecto es estimar un modelo de regresión lineal múltiple para analizar qué factores se asocian con el precio de los viajes de UberX. Para ello se utiliza una base de datos de viajes y una base climática, unidas por zona de origen y bloque horario.

La variable dependiente será el logaritmo natural del precio del viaje:

$$
\ln(price_i)
$$

Las variables explicativas consideradas son:

$$
\ln(distance_i), \quad rain\_amount_i, \quad noche_i, \quad humidity\_pct_i
$$

El modelo poblacional propuesto es:

$$
\ln(price_i)=\beta_0+\beta_1\ln(distance_i)+\beta_2rain\_amount_i+\beta_3noche_i+\beta_4humidity\_pct_i+u_i
$$

donde $u_i$ representa los factores no observados que también pueden afectar el precio del viaje, como disponibilidad de conductores, demanda específica por zona, tráfico, eventos cercanos u otros factores no incluidos en la base.

## Pregunta inicial de investigación

La pregunta que guía este proyecto es:

$$
\text{¿El precio de los viajes UberX depende únicamente de la distancia recorrida o también cambia de forma importante por condiciones climáticas y de horario?}
$$

En particular, se busca analizar si variables como la cantidad de lluvia, la humedad y el hecho de que el viaje ocurra de noche tienen una asociación estadísticamente relevante con el precio del viaje.

## Hipótesis inicial

Antes de estimar el modelo, la hipótesis inicial es que el clima sí afecta el precio de los viajes UberX. Intuitivamente, cuando las condiciones climáticas son menos cómodas, por ejemplo cuando llueve o hay mayor humedad, podría esperarse que más personas prefieran pedir un viaje en UberX en lugar de caminar, esperar transporte público o trasladarse al aire libre. Esto podría aumentar la demanda y, por lo tanto, asociarse con precios más altos.

De manera general, se espera que:

$$
\beta_1>0
$$

porque a mayor distancia recorrida, mayor debería ser el precio del viaje. También se espera que las variables climáticas tengan algún efecto sobre el precio, especialmente la lluvia y la humedad. Sin embargo, este punto será evaluado empíricamente a partir de los datos.

Es importante aclarar desde el inicio que el modelo permite estudiar asociaciones estadísticas entre variables, pero no implica necesariamente una relación causal directa.


## 1. Importación de librerías

Se importan las librerías necesarias para manipular la base de datos, realizar cálculos numéricos y estimar el modelo por Mínimos Cuadrados Ordinarios (MCO).


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

## 2. Carga y revisión inicial de la base

Se carga la base final de viajes UberX previamente limpiada y unida con información climática.  
La función `head()` permite visualizar las primeras observaciones, `shape` indica el número de filas y columnas, e `info()` muestra los tipos de datos y posibles valores faltantes.

Esta revisión es importante porque antes de estimar un modelo econométrico se debe verificar que las variables necesarias estén disponibles y que la base tenga suficientes observaciones.


In [ ]:
df = pd.read_excel("base_uberx_lluvia_noche_econometria.xlsx")

df.head()
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54848 entries, 0 to 54847
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   price        54848 non-null  float64
 1   distance     54848 non-null  float64
 2   rain_amount  54848 non-null  float64
 3   noche        54848 non-null  int64  
 4   ln_price     54848 non-null  float64
 5   ln_distance  54848 non-null  float64
 6   location     54848 non-null  object 
 7   hour         54848 non-null  object 
 8   rain_dummy   54848 non-null  int64  
 9   hora_pico    54848 non-null  int64  
 10  temp         54848 non-null  float64
 11  humidity     54848 non-null  float64
 12  wind         54848 non-null  float64
 13  clouds       54848 non-null  float64
dtypes: float64(9), int64(3), object(2)
memory usage: 5.9+ MB


## 3. Construcción de la variable de humedad en porcentaje

La variable original `humidity` está medida como proporción. Por ejemplo:

$$
0.75 = 75\%
$$

Para facilitar la interpretación, se construye la variable:

$$
humidity\_pct = humidity \times 100
$$

De esta forma, un aumento de una unidad en `humidity_pct` representa un aumento de un punto porcentual en la humedad, por ejemplo, pasar de 70% a 71%.


In [ ]:
df["humidity_pct"] = df["humidity"] * 100

df[["humidity", "humidity_pct"]].head()

,humidity,humidity_pct
0,0.570,57.0
1,0.630,63.0
2,0.810,81.0
3,0.905,90.5
4,0.895,89.5


## 4. Selección de variables para el modelo

Se construye una base específica para la regresión con las variables que forman parte del modelo final:

- `ln_price`: logaritmo natural del precio del viaje.
- `ln_distance`: logaritmo natural de la distancia.
- `rain_amount`: cantidad registrada de lluvia en la zona y hora del viaje.
- `noche`: variable dummy que vale 1 si el viaje ocurrió de noche y 0 en caso contrario.
- `humidity_pct`: humedad expresada en puntos porcentuales.

Se eliminan observaciones con valores faltantes mediante `dropna()` para estimar el modelo con una muestra completa.


In [ ]:
df_modelo = df[[
    "ln_price",
    "ln_distance",
    "rain_amount",
    "noche",
    "humidity_pct"
]].dropna().copy()

df_modelo.head()

,ln_price,ln_distance,rain_amount,noche,humidity_pct
0,2.014903,0.104360,0.0000,0,57.0
1,2.140066,0.908259,0.0000,0,63.0
2,2.251292,1.078410,0.0000,0,81.0
3,2.251292,0.148420,0.0000,0,90.5
4,2.251292,0.982078,0.2443,1,89.5


## 5. Estimación del modelo por Mínimos Cuadrados Ordinarios

El modelo estimado es:

$$
\ln(price_i)=\beta_0+\beta_1\ln(distance_i)+\beta_2rain\_amount_i+\beta_3noche_i+\beta_4humidity\_pct_i+u_i
$$

La variable dependiente es \(Y=\ln(price)\), mientras que la matriz de variables explicativas \(X\) contiene una constante y las variables independientes del modelo.

El método de Mínimos Cuadrados Ordinarios estima los coeficientes que minimizan la suma de los cuadrados de los residuos.


In [ ]:
Y = df_modelo["ln_price"]

X = df_modelo[[
    "ln_distance",
    "rain_amount",
    "noche",
    "humidity_pct"
]]

X = sm.add_constant(X)

modelo_final = sm.OLS(Y, X).fit()

print(modelo_final.summary())

                            OLS Regression Results                            
Dep. Variable:               ln_price   R-squared:                       0.543
Model:                            OLS   Adj. R-squared:                  0.543
Method:                 Least Squares   F-statistic:                 1.628e+04
Date:                Mon, 15 Jun 2026   Prob (F-statistic):               0.00
Time:                        03:00:14   Log-Likelihood:                 24934.
No. Observations:               54848   AIC:                        -4.986e+04
Df Residuals:                   54843   BIC:                        -4.981e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            2.0879      0.004    512.594   

## 6. Tabla de coeficientes

La siguiente tabla resume los elementos principales de la inferencia individual para cada estimador:

- **Coeficiente:** valor estimado de cada \(\hat{\beta}\).
- **Error estándar:** medida de la variabilidad del estimador.
- **t calculada:** estadístico utilizado para contrastar hipótesis sobre cada coeficiente.
- **p-value:** probabilidad asociada al estadístico \(t\), útil para decidir si se rechaza o no la hipótesis nula.

Para cada coeficiente se puede plantear:

$$
H_0:\beta_j=0
$$

frente a:

$$
H_1:\beta_j\neq 0
$$

Si el p-value es menor que el nivel de significancia elegido, por ejemplo 5%, se rechaza \(H_0\).


In [ ]:
tabla_coeficientes = pd.DataFrame({
    "Coeficiente": modelo_final.params,
    "Error estándar": modelo_final.bse,
    "t calculada": modelo_final.tvalues,
    "p-value": modelo_final.pvalues
})

tabla_coeficientes

,Coeficiente,Error estándar,t calculada,p-value
const,2.087886,0.004073,512.593574,0.000000
ln_distance,0.274719,0.001077,255.149927,0.000000
rain_amount,-0.005166,0.005126,-1.007830,0.313541
noche,0.001120,0.001359,0.824168,0.409848
humidity_pct,-0.000108,0.000052,-2.084039,0.037161


## Lectura inicial de la tabla de coeficientes

En la tabla de coeficientes se observa que `ln_distance` es la variable con mayor significancia estadística, ya que su p-value es prácticamente cero. Esto indica que la distancia recorrida sí está asociada con el precio del viaje.

Por otro lado, `rain_amount` y `noche` presentan p-values mayores a 0.05, por lo que no se rechaza la hipótesis nula de que sus coeficientes sean iguales a cero. Esto significa que, en este modelo, no existe evidencia estadística suficiente para afirmar que la cantidad de lluvia o el horario nocturno afecten de manera significativa el precio de UberX.

Finalmente, `humidity_pct` sí presenta un p-value menor a 0.05, por lo que se considera estadísticamente significativa. Sin embargo, su coeficiente es muy pequeño, por lo que su efecto económico debe interpretarse con cautela.


## Conclusiones de las pruebas de hipótesis para cada \(\beta\)

Para realizar inferencia individual sobre cada coeficiente del modelo, se considera un nivel de significancia de:

$$
\alpha = 0.05
$$

Para cada variable explicativa se plantea la prueba:

$$
H_0:\beta_j=0
$$

contra:

$$
H_1:\beta_j\neq 0
$$

La regla de decisión es:

- Si \(p\text{-value}<0.05\), se rechaza \(H_0\).
- Si \(p\text{-value}\geq 0.05\), no se rechaza \(H_0\).

### Prueba para \(\beta_1\): efecto de \(\ln(distance)\)

Para la variable `ln_distance`, el coeficiente estimado fue:

$$
\hat{\beta}_1=0.274719
$$

y su p-value fue prácticamente:

$$
p\text{-value}=0.000000
$$

Como:

$$
0.000000<0.05
$$

se rechaza la hipótesis nula:

$$
H_0:\beta_1=0
$$

Por lo tanto, existe evidencia estadística suficiente para afirmar que la distancia recorrida tiene una relación significativa con el precio del viaje UberX. Además, como el coeficiente es positivo, se concluye que a mayor distancia recorrida, mayor es el precio esperado del viaje, manteniendo constantes las demás variables.

En términos económicos, si la distancia aumenta en 1%, el precio esperado aumenta aproximadamente en:

$$
0.274719\%
$$

ceteris paribus.

### Prueba para \(\beta_2\): efecto de `rain_amount`

Para la variable `rain_amount`, el coeficiente estimado fue:

$$
\hat{\beta}_2=-0.005166
$$

y su p-value fue:

$$
p\text{-value}=0.313541
$$

Como:

$$
0.313541>0.05
$$

no se rechaza la hipótesis nula:

$$
H_0:\beta_2=0
$$

Por lo tanto, no existe evidencia estadística suficiente para afirmar que la cantidad registrada de lluvia tenga una relación significativa con el precio del viaje UberX, manteniendo constantes la distancia, la noche y la humedad.

Aunque el coeficiente estimado es negativo, no debe interpretarse como un efecto relevante, ya que estadísticamente no se puede distinguir de cero al nivel de significancia del 5%.

### Prueba para \(\beta_3\): efecto de `noche`

Para la variable `noche`, el coeficiente estimado fue:

$$
\hat{\beta}_3=0.001120
$$

y su p-value fue:

$$
p\text{-value}=0.409848
$$

Como:

$$
0.409848>0.05
$$

no se rechaza la hipótesis nula:

$$
H_0:\beta_3=0
$$

Por lo tanto, no existe evidencia estadística suficiente para afirmar que un viaje nocturno tenga una relación significativa con el precio esperado de UberX, manteniendo constantes la distancia, la lluvia y la humedad.

Aunque el coeficiente indicaría que los viajes nocturnos tienen un precio esperado aproximadamente 0.1120% mayor que los viajes diurnos, esta diferencia no es estadísticamente significativa.

### Prueba para \(\beta_4\): efecto de `humidity_pct`

Para la variable `humidity_pct`, el coeficiente estimado fue:

$$
\hat{\beta}_4=-0.000108
$$

y su p-value fue:

$$
p\text{-value}=0.037161
$$

Como:

$$
0.037161<0.05
$$

se rechaza la hipótesis nula:

$$
H_0:\beta_4=0
$$

Por lo tanto, existe evidencia estadística suficiente para afirmar que la humedad tiene una relación significativa con el precio del viaje UberX, manteniendo constantes la distancia, la lluvia y si el viaje fue de noche.

Sin embargo, aunque la variable es estadísticamente significativa, su efecto económico es muy pequeño. El coeficiente indica que si la humedad aumenta en un punto porcentual, por ejemplo de 70% a 71%, el precio esperado del viaje disminuye aproximadamente en:

$$
0.0108\%
$$

ceteris paribus.

### Conclusión de las pruebas individuales

A partir de las pruebas de hipótesis individuales, se concluye que las variables estadísticamente significativas al 5% son:

$$
\ln(distance)
$$

y

$$
humidity\_pct
$$

En cambio, las variables:

$$
rain\_amount
$$

y

$$
noche
$$

no resultaron estadísticamente significativas al 5%.

Esto significa que, dentro del modelo estimado, el precio de UberX está explicado principalmente por la distancia recorrida. La humedad también aparece como significativa, aunque con un efecto práctico muy pequeño. Por otro lado, no se encontró evidencia suficiente para afirmar que la lluvia registrada o el horario nocturno modifiquen de forma significativa el precio del viaje.


## 7. Ecuación estimada

Con los coeficientes obtenidos se escribe la ecuación estimada del modelo:

$$
\widehat{\ln(price_i)}
=
\hat{\beta}_0+
\hat{\beta}_1\ln(distance_i)+
\hat{\beta}_2rain\_amount_i+
\hat{\beta}_3noche_i+
\hat{\beta}_4humidity\_pct_i
$$

Esta ecuación permite interpretar el efecto parcial de cada variable explicativa sobre el logaritmo del precio, manteniendo constantes las demás variables, es decir, bajo el supuesto ceteris paribus.


In [ ]:
b0 = modelo_final.params["const"]
b1 = modelo_final.params["ln_distance"]
b2 = modelo_final.params["rain_amount"]
b3 = modelo_final.params["noche"]
b4 = modelo_final.params["humidity_pct"]

print("Ecuación estimada:")
print(f"ln(price) = {b0:.6f} + {b1:.6f}ln(distance) + {b2:.6f}rain_amount + {b3:.6f}noche + {b4:.6f}humidity_pct")

Ecuación estimada:
ln(price) = 2.087886 + 0.274719ln(distance) + -0.005166rain_amount + 0.001120noche + -0.000108humidity_pct


## 8. Bondad de ajuste y varianza estimada del error

En esta sección se calculan algunos estadísticos solicitados para evaluar el modelo:

La suma de cuadrados del error es:

$$
SCE=\sum_{i=1}^{n}\hat{u}_i^2
$$

La varianza estimada del error es:

$$
\hat{\sigma}^2=\frac{\sum_{i=1}^{n}\hat{u}_i^2}{n-k-1}
$$

donde \(n\) es el número de observaciones y \(k\) es el número de variables independientes.

El error estándar de la regresión es:

$$
e.e.r.=\sqrt{\hat{\sigma}^2}
$$

Además, se reportan \(R^2\) y \(R^2\) ajustada. El \(R^2\) mide la proporción de la variación de la variable dependiente explicada por el modelo, mientras que el \(R^2\) ajustado castiga la inclusión de variables adicionales.


In [ ]:
residuos = modelo_final.resid

n = len(df_modelo)
k = 4

SCE = sum(residuos**2)

sigma2_hat = SCE / (n - k - 1)

eer = np.sqrt(sigma2_hat)

print("Número de observaciones:", n)
print("Suma de cuadrados del error:", SCE)
print("Sigma cuadrada estimada:", sigma2_hat)
print("Error estándar de la regresión:", eer)
print("R cuadrada:", modelo_final.rsquared)
print("R cuadrada ajustada:", modelo_final.rsquared_adj)

Número de observaciones: 54848
Suma de cuadrados del error: 1293.6953524493663
Sigma cuadrada estimada: 0.0235890697527372
Error estándar de la regresión: 0.153587335912624
R cuadrada: 0.5428242124792699
R cuadrada ajustada: 0.5427908681481779


## 9. Cálculo matricial de los estimadores

Para relacionar el procedimiento con el desarrollo teórico visto en clase, también se calculan los estimadores mediante la expresión matricial de MCO:

$$
\hat{\beta}=(X'X)^{-1}X'Y
$$

Este cálculo debe coincidir con los coeficientes obtenidos mediante `statsmodels`, ya que ambos aplican el mismo método de estimación.


In [ ]:
X_matriz = X.values
Y_matriz = Y.values.reshape(-1, 1)

beta_hat = np.linalg.inv(X_matriz.T @ X_matriz) @ (X_matriz.T @ Y_matriz)

tabla_beta_matriz = pd.DataFrame(
    beta_hat,
    index=["Intercepto", "ln_distance", "rain_amount", "noche", "humidity_pct"],
    columns=["Beta estimado por matriz"]
)

tabla_beta_matriz

,Beta estimado por matriz
Intercepto,2.087886
ln_distance,0.274719
rain_amount,-0.005166
noche,0.001120
humidity_pct,-0.000108


## 10. Varianza de los estimadores y errores estándar

La matriz de varianzas y covarianzas de los estimadores se calcula como:

$$
Var(\hat{\beta})=\hat{\sigma}^2(X'X)^{-1}
$$

La varianza de cada estimador se encuentra en la diagonal principal de esta matriz.  
El error estándar de cada coeficiente se obtiene como:

$$
e.e.(\hat{\beta}_j)=\sqrt{Var(\hat{\beta}_j)}
$$

Con estos errores estándar se calculan los estadísticos \(t\):

$$
t=\frac{\hat{\beta}_j}{e.e.(\hat{\beta}_j)}
$$

Estos valores permiten realizar inferencia estadística sobre los parámetros del modelo.


In [ ]:
var_beta = sigma2_hat * np.linalg.inv(X_matriz.T @ X_matriz)

ee_beta = np.sqrt(np.diag(var_beta))

tabla_manual = pd.DataFrame({
    "Beta estimado": beta_hat.flatten(),
    "Varianza de beta": np.diag(var_beta),
    "Error estándar": ee_beta,
    "t calculada": beta_hat.flatten() / ee_beta,
    "p-value": modelo_final.pvalues.values
}, index=["Intercepto", "ln_distance", "rain_amount", "noche", "humidity_pct"])

tabla_manual

,Beta estimado,Varianza de beta,Error estándar,t calculada,p-value
Intercepto,2.087886,1.659080e-05,0.004073,512.593574,0.000000
ln_distance,0.274719,1.159273e-06,0.001077,255.149927,0.000000
rain_amount,-0.005166,2.627299e-05,0.005126,-1.007830,0.313541
noche,0.001120,1.847929e-06,0.001359,0.824168,0.409848
humidity_pct,-0.000108,2.690125e-09,0.000052,-2.084039,0.037161


## 11. Interpretación específica de los resultados y conclusión

A partir de la estimación del modelo final se obtuvo la siguiente ecuación:

$$
\widehat{\ln(price_i)}
=
2.087886
+
0.274719\ln(distance_i)
-
0.005166rain\_amount_i
+
0.001120noche_i
-
0.000108humidity\_pct_i
$$

El coeficiente de `ln_distance` fue:

$$
\hat{\beta}_1=0.274719
$$

Como tanto el precio como la distancia están en logaritmos, este coeficiente se interpreta como una elasticidad. Por lo tanto, si la distancia del viaje aumenta en 1%, el precio esperado del viaje aumenta aproximadamente en 0.274719%, manteniendo constantes la cantidad de lluvia, si el viaje fue de noche y la humedad. Esta variable resultó altamente significativa, pues su p-value fue prácticamente 0. Esto confirma que la distancia recorrida es el principal factor explicativo del precio de los viajes UberX dentro de este modelo.

El coeficiente de `rain_amount` fue:

$$
\hat{\beta}_2=-0.005166
$$

Como la variable dependiente está en logaritmo, este coeficiente se interpreta aproximadamente como un cambio porcentual. Así, si la cantidad registrada de lluvia aumenta en una unidad, el precio esperado del viaje disminuye aproximadamente en 0.5166%, manteniendo constantes la distancia, el horario nocturno y la humedad. Sin embargo, su p-value fue 0.313541, por lo que no es estadísticamente significativa al 5%. Esto quiere decir que, con esta muestra, no hay evidencia suficiente para afirmar que la cantidad de lluvia tenga un efecto relevante sobre el precio de UberX.

El coeficiente de `noche` fue:

$$
\hat{\beta}_3=0.001120
$$

La variable `noche` es una variable dummy, por lo que compara viajes nocturnos contra viajes diurnos. Su interpretación aproximada es que, si el viaje ocurrió de noche, el precio esperado aumenta en 0.1120% respecto a un viaje de día, manteniendo constantes la distancia, la lluvia y la humedad. No obstante, su p-value fue 0.409848, por lo que tampoco es estadísticamente significativa. En consecuencia, no se encontró evidencia suficiente para sostener que viajar de noche modifique de manera importante el precio esperado de UberX.

El coeficiente de `humidity_pct` fue:

$$
\hat{\beta}_4=-0.000108
$$

Esta variable mide la humedad en puntos porcentuales. Por ejemplo, pasar de 70% a 71% de humedad representa un aumento de una unidad en `humidity_pct`. La interpretación del coeficiente es que, si la humedad aumenta en un punto porcentual, el precio esperado del viaje disminuye aproximadamente en 0.0108%, manteniendo constantes la distancia, la lluvia y si el viaje fue de noche.

A diferencia de la lluvia y la variable nocturna, la humedad sí resultó estadísticamente significativa al 5%, ya que su p-value fue 0.037161. Sin embargo, aunque el efecto es estadísticamente significativo, su magnitud económica es muy pequeña. Por ejemplo, si la humedad aumentara 10 puntos porcentuales, el precio esperado disminuiría aproximadamente en:

$$
10(0.0108\%)=0.108\%
$$

Este cambio es muy reducido en términos prácticos.

En cuanto a la bondad de ajuste, el modelo obtuvo una \(R^2\) aproximada de 0.543. Esto significa que el modelo explica alrededor del 54.3% de la variación observada en el logaritmo del precio de los viajes UberX. Para un modelo con pocas variables explicativas, este nivel de ajuste es razonable, aunque también indica que existe una parte importante del precio que depende de factores no incluidos en el modelo.

## Respuesta a la pregunta inicial

La pregunta inicial era si el precio de UberX dependía únicamente de la distancia o si también cambiaba de forma importante por condiciones climáticas y de horario.

Con base en los resultados, la respuesta es parcial. La distancia sí tiene un efecto claro, positivo y estadísticamente significativo sobre el precio. En cambio, las variables de lluvia y horario nocturno no resultaron estadísticamente significativas. La humedad sí fue significativa, pero su efecto estimado fue muy pequeño.

Por lo tanto, la hipótesis inicial de que el clima afecta mucho el precio de UberX no se sostiene completamente. Los datos muestran que la distancia recorrida es el factor más importante del modelo, mientras que las variables climáticas tienen un papel mucho más limitado. La humedad aparece como significativa, pero no con una magnitud suficientemente grande como para decir que cambia de manera importante el precio del viaje.

## Correlación no implica causalidad

Un punto importante del análisis es que los coeficientes estimados deben interpretarse como asociaciones estadísticas y no como efectos causales directos. Por ejemplo, la humedad resultó significativa y con signo negativo, pero eso no significa que un aumento en la humedad cause directamente una disminución en el precio de UberX.

Este resultado puede deberse a otros factores no observados. Por ejemplo, la humedad puede estar relacionada con ciertas zonas, horarios, días específicos, patrones de movilidad, tráfico, disponibilidad de conductores o niveles de demanda que no están incluidos explícitamente en el modelo. Si alguno de esos factores también influye en el precio, entonces el coeficiente de humedad puede estar capturando parte de esas relaciones indirectas.

Además, la intuición inicial podría sugerir lo contrario: si hay más humedad o condiciones climáticas incómodas, se podría esperar que más personas pidan UberX, lo cual podría elevar los precios. Sin embargo, los datos arrojaron una asociación negativa y muy pequeña. Esto muestra justamente por qué en econometría no basta con tener una intuición razonable; es necesario contrastarla con datos y, aun así, interpretar los resultados con cuidado.

En conclusión, el modelo permite decir que existe una asociación estadística entre las variables incluidas y el precio, especialmente en el caso de la distancia. Pero no permite afirmar que la lluvia, la noche o la humedad causen cambios en el precio de forma directa. Para hacer afirmaciones causales más fuertes sería necesario contar con más información, por ejemplo medidas directas de demanda, disponibilidad de conductores, tráfico, zona específica, eventos o el algoritmo de precios de la plataforma.
